# Bloom Text 0.2 — continued pretraining

This notebook runs **one** continued-pretraining session of Bloom Text 0.2 (54,807,040 parameters) and opens a review-only pull request against [`Centauri-Bloom/Bloom-Text-0.2`](https://huggingface.co/Centauri-Bloom/Bloom-Text-0.2). It does not overwrite `main` until a maintainer merges that PR. It does not train Chat 0.2, and it does not use private contributor datasets.

Use this on **Kaggle** with a GPU T4 x2 accelerator (the usual session setting). It also runs on Colab (T4) or a local CUDA/MPS machine. A full 10,000-step session takes about 90 minutes on T4 x2 and uses about 1.5–2 hours of a free Kaggle account's weekly 30 GPU-hour quota. Each session exposes 163,840,000 tokens (10,000 steps × 16,384 tokens/step).

## Before you run

1. Create a free [Kaggle](https://www.kaggle.com/) account and complete phone verification so GPU accelerators are available. Enable Internet in the notebook session.
2. Create a free [Hugging Face](https://huggingface.co/join) account. Make a **write** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). In Kaggle, add it as a secret named exactly `HF_TOKEN` (Add-ons → Secrets). Do not paste the token into the notebook or into a GitHub issue.
3. **Set `PARENT_HF_REVISION` in the settings cell** to the latest open Bloom-Text-0.2 checkpoint. Open [the model's discussions](https://huggingface.co/Centauri-Bloom/Bloom-Text-0.2/discussions), find the newest open training PR (for example `refs/pr/4`), and use that. If that PR has already been merged, use `main`. Do not start from an older parent if a newer checkpoint exists.
4. Comment on the tracking GitHub issue before you start, so two people do not train from the same parent at once. Parallel candidates are not additive.
5. On Kaggle: File → Import Notebook, choose this file, set the accelerator to **GPU T4 x2**, confirm the `HF_TOKEN` secret is attached, then **Save & Run All**.

If the session disconnects mid-run, set `RECOVER_RUN_DIR` to the run folder under `/kaggle/working/centauri-bloom-runs` and run again. If training finished but upload failed, recover and re-run the upload; do not retrain.

Candidate weights are submitted as a Hugging Face PR for review. Never put secrets in the notebook, in logs, or in issues.


In [ ]:
#@title 1. Choose training settings
# Bloom Text 0.2 continued pretraining. Leave MODEL, DATASET, STEPS, and STAGE as they are
# unless a maintainer asked you to change them.
MODEL = "50M"
DATASET = "FineWeb-Edu + literature + TinyStories"
STEPS = 10000
STAGE = "pretrain-continue"
# Checkout the merged trainer on main. Change this only if a maintainer names another branch.
REPO_REF = "main"
# Review FineWeb-Edu (ODC-By 1.0; underlying web rights vary), public-domain-v1, and TinyStories (CDLA-Sharing-1.0) before confirming.
CONFIRM_DATA_PERMISSION = True
# Optional: resume a previous unified run, including mid-run checkpoints, from this folder.
RECOVER_RUN_DIR = ""
PARENT_RUN_DIR = ""
PARENT_HF_REPO = "Centauri-Bloom/Bloom-Text-0.2"

# ---------------------------------------------------------------------------
# SET THIS before you click Save & Run All.
# Use the latest open checkpoint on
# https://huggingface.co/Centauri-Bloom/Bloom-Text-0.2/discussions
# Example: "refs/pr/4" for discussion/PR 4, or "main" if that checkpoint is already merged.
# Do not leave a stale revision if a newer one is open.
# ---------------------------------------------------------------------------
PARENT_HF_REVISION = "refs/pr/4"

if not CONFIRM_DATA_PERMISSION:
    raise ValueError("Confirm dataset permission before running.")
if MODEL != "50M" or STAGE != "pretrain-continue":
    raise ValueError("This notebook is only for Bloom Text 0.2 continued pretraining (MODEL=50M, STAGE=pretrain-continue).")
if not str(PARENT_HF_REVISION).strip() and not str(PARENT_RUN_DIR).strip():
    raise ValueError("Set PARENT_HF_REVISION to the latest open Bloom-Text-0.2 checkpoint (for example refs/pr/4), or PARENT_RUN_DIR to a completed local run.")
print("50M continued pretraining loads Bloom-Text-0.2 (PARENT_HF_REVISION or PARENT_RUN_DIR) and uploads a new PR only. Main is not overwritten.")
print("Settings: DATASET=FineWeb-Edu + literature + TinyStories, STEPS=10000, PARENT_HF_REPO=Centauri-Bloom/Bloom-Text-0.2, PARENT_HF_REVISION=%s." % PARENT_HF_REVISION)
print("This uses a 1e-4 cosine schedule with warmup, batch 8, accumulation 4 (16,384 tokens/step). Optimizer state is not on Hugging Face; a new AdamW starts, and mid-run RECOVER_RUN_DIR still restores partial-optimizer.pt.")
print("Confirm PARENT_HF_REVISION is the latest open checkpoint: https://huggingface.co/Centauri-Bloom/Bloom-Text-0.2/discussions")
print("Selected:", MODEL, "|", STAGE, "|", DATASET, "|", STEPS, "optimizer step attempts")


In [ ]:
#@title 2. Set up the environment
import os,sys,subprocess,json,shutil
from pathlib import Path
from datetime import datetime,timezone

def find_repo(start):
    cur = Path(start).resolve()
    for p in [cur, *cur.parents]:
        if (p/"pyproject.toml").is_file() and (p/"bloom"/"model.py").is_file():
            return p
    return None

IN_KAGGLE = Path("/kaggle/working").is_dir()
IN_COLAB = False
if not IN_KAGGLE:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
IN_LOCAL = not IN_KAGGLE and not IN_COLAB
os.environ["HF_HUB_DISABLE_XET"] = "1"

def checked(args):
    p = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout, flush=True)
    if p.returncode:
        raise RuntimeError(f"Setup failed (exit {p.returncode}). See full diagnostic above.")

if IN_KAGGLE:
    REPO = Path("/kaggle/working/centauri-bloom")
elif IN_COLAB:
    REPO = Path("/content/centauri-bloom")
else:
    REPO = find_repo(Path.cwd())
    if REPO is None:
        raise RuntimeError("Open this notebook from a centauri-bloom checkout, or run it from the repo root.")

REF = (REPO_REF if "REPO_REF" in dir() else "main") or "main"
REF = str(REF).strip() or "main"
if not IN_LOCAL:
    if not (REPO/".git").exists():
        checked(["git", "clone", "--depth", "1", "--branch", REF, "https://github.com/InterstellarNerd8760/centauri-bloom.git", str(REPO)])
    else:
        checked(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REF])
        checked(["git", "-C", str(REPO), "checkout", "-B", REF, "FETCH_HEAD"])
os.chdir(REPO)
# Preserve Colab/Kaggle CUDA-enabled PyTorch and its matching dependencies.
checked([sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."])
checked([sys.executable, "-m", "pip", "install", "tokenizers", "datasets", "huggingface_hub", "ipywidgets", "safetensors", "gguf"])

from bloom.notebook_workflow import submit_candidate, pick_device, preflight, prepare_recovered_run, upload_base_revision
from bloom.trainer import target_repo
from bloom.trainer_data import RECIPES

DEVICE = pick_device()
if IN_KAGGLE:
    BASE = Path("/kaggle/working/centauri-bloom-runs")
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/centauri-bloom-runs")
else:
    BASE = REPO / "runs"
BASE.mkdir(parents=True, exist_ok=True)
slug = (MODEL + "-pretrain-continue-" + DATASET).lower().replace(" + ", "-").replace(" ", "-")
if RECOVER_RUN_DIR:
    try:
        OUT_DIR = prepare_recovered_run(RECOVER_RUN_DIR, BASE)
        print("Recovered run:", OUT_DIR)
    except FileNotFoundError:
        OUT_DIR = Path(RECOVER_RUN_DIR)
        print("No completed run at RECOVER_RUN_DIR yet; a mid-run resume will use", OUT_DIR)
else:
    OUT_DIR = BASE / ("bloom-" + slug + "-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
UPLOAD_ONLY = RECOVER_RUN_DIR and (OUT_DIR / "metrics.json").is_file()
if not IN_LOCAL and DEVICE != "cuda" and not UPLOAD_ONLY:
    raise RuntimeError("Enable a GPU runtime (T4 x2 on Kaggle, T4 on Colab). Upload-only recovery does not need a GPU.")
if IN_LOCAL and DEVICE == "cpu" and not UPLOAD_ONLY:
    print("Warning: a full 50M run on CPU will be very slow. MPS or CUDA is strongly preferred.")
if DEVICE == "cuda" and shutil.which("nvidia-smi") and not UPLOAD_ONLY:
    checked(["nvidia-smi"])
preflight(DEVICE if not UPLOAD_ONLY else "cpu")
if DATASET not in RECIPES:
    raise ValueError("Invalid dataset.")
if type(STEPS) is not int or not 1 <= STEPS <= 10000:
    raise ValueError("Invalid step budget.")
HF_REPO_ID = target_repo(MODEL, DATASET, stage=STAGE)
print("Environment:", "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local", "| device:", DEVICE, "| output:", OUT_DIR)
print("Upload target:", HF_REPO_ID)
if UPLOAD_ONLY:
    print("Completed candidate found; training will be skipped and the existing weights uploaded.")


In [ ]:
#@title 3. Authorize Hugging Face
from getpass import getpass
from bloom.notebook_workflow import auth_client
try:
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    elif IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    else:
        token = os.environ.get("HF_TOKEN")
except Exception:
    token = None
if not token:
    token = getpass("Hugging Face write token: ")
AUTH_API = auth_client(HF_REPO_ID, token, require_repo=(HF_REPO_ID == "Centauri-Bloom/Bloom-Text-0.1"))
del token
print("Authenticated. Candidate PR only. Target:", HF_REPO_ID)


In [ ]:
#@title 4. Load the selected model, prepare data, train and evaluate
if AUTH_API is None:
    raise RuntimeError("Authorize upload first.")
# Train in a fresh process, avoiding stale notebook imports and Dynamo state.
env = os.environ.copy()
env["HF_TOKEN"] = AUTH_API.token
payload = json.dumps(dict(
    recipe=DATASET, steps=STEPS, repo_dir=str(REPO), out_dir=str(OUT_DIR),
    recover=bool(RECOVER_RUN_DIR), model_size=MODEL, device=DEVICE,
    parent_run_dir=PARENT_RUN_DIR or None,
    parent_hf_repo=PARENT_HF_REPO or None,
    parent_hf_revision=PARENT_HF_REVISION or None,
    stage=STAGE,
))
script = ("import json,sys,os; from huggingface_hub import HfApi; from bloom.notebook_workflow import preflight; "
          "from bloom.trainer import run_training; args=json.loads(sys.argv[1]); preflight(args['device']); "
          "run_training(HfApi(token=os.environ['HF_TOKEN']), **args)")
try:
    result = subprocess.run([sys.executable, "-u", "-c", script, payload], env=env)
finally:
    env.pop("HF_TOKEN", None)
if result.returncode:
    raise RuntimeError("Training failed. See diagnostic above; saved files remain in the output directory. Set RECOVER_RUN_DIR to that folder to resume.")
metrics = json.loads((OUT_DIR / "metrics.json").read_text())
print("Wrote", OUT_DIR / "metrics.json", "and", OUT_DIR / "evaluation.json")


In [ ]:
#@title 5. Export GGUF and automatically submit the completed candidate
if AUTH_API is None:
    raise RuntimeError("Authorize upload first.")
metrics = json.loads((OUT_DIR / "metrics.json").read_text())
if (metrics["additional_steps"] != STEPS or metrics["dataset"] != DATASET or metrics.get("model_size", "14M") != MODEL):
    raise RuntimeError("Run settings differ from completed candidate.")
if HF_REPO_ID != metrics.get("target_repo", HF_REPO_ID):
    raise RuntimeError("Upload target differs from the completed run.")
if HF_REPO_ID != "Centauri-Bloom/Bloom-Text-0.2":
    raise RuntimeError("This notebook must target Bloom-Text-0.2, not a chat repo or Bloom-Text-0.1.")
upload_revision = upload_base_revision(metrics)
submit_candidate(AUTH_API, HF_REPO_ID, OUT_DIR, STEPS, upload_revision, metrics["run_name"])
print("Review and merge the candidate in Hugging Face before the next run.")
print("This PR is additive; it does not overwrite the existing Text-0.2 checkpoint until you merge it.")
print("If upload failed after training finished, set RECOVER_RUN_DIR to", OUT_DIR, "and re-run, or use notebooks/bloom-upload-recovery.ipynb. Do not retrain.")
